# Session 5: Inventing a floor plan

**259.075 Data-integrated Algorithmic Design Processes II** · Session 5 of 7

Last session ended with a map of shapes and a question: if you can walk through that map, can
you also stand somewhere in it and say *here shall be a building*?

With the autoencoder from last week, not dependably. The reason why is the whole content of
today, and the repair turns out to be small, old and slightly surprising.

In [ ]:
import sys
import numpy as np
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

sys.path.insert(0, "../data")
from autoencoder import Autoencoder
from vae import VAE, GRUPPEN, ENGE

bilder = np.load("../data/grundrisse.npy")
tabelle = np.genfromtxt("../data/grundrisse.csv", delimiter=",", names=True,
                        dtype=None, encoding="utf-8")

ae = Autoencoder(8)
ae.load_state_dict(torch.load("../data/ae_8.pt"))
ae.eval()

vae = VAE()
vae.load_state_dict(torch.load("../data/vae_8.pt"))
vae.eval()

bedingt = VAE(klassen=len(GRUPPEN))
bedingt.load_state_dict(torch.load("../data/cvae_8.pt"))
bedingt.eval()

codes_ae = np.load("../data/codes.npz")["code_8"]
daten_vae = np.load("../data/vae_codes.npz")
codes_vae, gruppen = daten_vae["code"], daten_vae["gruppe"]
print(bilder.shape, "outlines,", ENGE, "numbers in the middle")

## Where am I allowed to point?

To ask a decoder for a building you have to hand it eight numbers, and immediately the question
is which eight. The autoencoder never answered that. It was trained to reproduce buildings, and
it put each one wherever was convenient. Look at what that looks like.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "autoencoder: first two of the eight numbers", "variational autoencoder: the same two"))
fig.add_trace(go.Scatter(x=codes_ae[:, 0], y=codes_ae[:, 1], mode="markers",
                         marker=dict(size=4, color="#4c78a8", opacity=0.6),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=codes_vae[:, 0], y=codes_vae[:, 1], mode="markers",
                         marker=dict(size=4, color="crimson", opacity=0.6),
                         showlegend=False), row=1, col=2)
fig.update_layout(height=430, title="the same 1071 buildings, in two different code spaces")
fig.show()

for name, c in (("autoencoder", codes_ae), ("vae", codes_vae)):
    print(f"{name:12s} mean {c.mean():+.2f}, spread {c.std():.2f}, "
          f"range {c.min():+.1f} to {c.max():+.1f}")

The autoencoder's codes sit in a cloud of no particular shape, at no particular scale, with
holes in it. Nothing forced them anywhere else. The variational autoencoder's codes are centred
on zero on a far more controlled scale, because the second term in its loss pulls the encoder
towards a standard normal distribution.

That difference is not cosmetic. The variational autoencoder comes with a trained rule for where
to draw codes from; the plain autoencoder comes with none. The next cell asks both decoders for sixteen buildings. The autoencoder comes with no
rule for choosing the eight numbers, so we draw uniformly inside the box its codes happen to
occupy. That is one reasonable choice among several, not a guaranteed best case.

In [ ]:
def zeige(stapel, titel, spalten=8):
    zeilen = int(np.ceil(len(stapel) / spalten))
    fig = make_subplots(rows=zeilen, cols=spalten, horizontal_spacing=0.01,
                        vertical_spacing=0.03)
    for n, bild in enumerate(stapel):
        r, c = divmod(n, spalten)
        fig.add_trace(go.Heatmap(zmin=0, zmax=1, z=bild[::-1], colorscale="Greys", reversescale=True,
                                 showscale=False), row=r + 1, col=c + 1)
    fig.update_layout(height=120 * zeilen + 80, title=titel)
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    return fig


wuerfel = torch.Generator().manual_seed(1)
lo, hi = codes_ae.min(axis=0), codes_ae.max(axis=0)
gezogen = np.random.default_rng(1).uniform(lo, hi, size=(16, 8)).astype("float32")
with torch.no_grad():
    aus_ae = torch.sigmoid(ae.decoder(torch.tensor(gezogen))).numpy().reshape(-1, 32, 32)
zeige(aus_ae, "sixteen points picked inside the autoencoder's code box").show()

Some of those are buildings. Several are hourglasses, dents and blobs that no surveyor ever
recorded. The decoder was never asked what lies between its training examples, so between them
it does whatever the weights happen to do.

> A decoder is a function, not an oracle. It answers everywhere, including where it knows
> nothing, and it does not mark the difference.

## The repair

The change is one line of arithmetic in the encoder and one extra term in the loss.

Instead of producing a point, the encoder produces a *mean* and a *spread* for each of the eight
numbers, and the code actually used is drawn from that. So a building no longer occupies a
point, it occupies a small cloud, and the decoder has to cope with the whole cloud. Clouds of
nearby buildings overlap, and the gaps close.

The second term pulls all the clouds towards the same place, a standard normal distribution
centred on zero with spread one. It is called the Kullback-Leibler term, and here it does one
job: it makes the answer to *where am I allowed to point* be "anywhere near the origin". The
formula is in the file, `data/vae.py`, and is not examined.

> The two terms pull against each other. Reconstruction wants the clouds far apart and tight;
> the second term wants them overlapping at the origin. What you get is the compromise.

Now the sampling actually makes sense: draw from a standard normal, decode. Write it.

In [ ]:
def ziehe(modell, n=16, temperatur=1.0, c=None, wuerfel=None):
    """n outlines drawn from the model. temperatur scales the spread of the draw."""

    # YOUR TURN: two lines. Draw n codes from a normal distribution whose spread is
    # `temperatur`, then push them through the decoder and squash with torch.sigmoid.
    raise NotImplementedError("sampling missing")

    return torch.sigmoid(roh).detach().numpy().reshape(-1, 32, 32)


wuerfel = torch.Generator().manual_seed(1)
zeige(ziehe(vae, 16, wuerfel=wuerfel),
      "sixteen buildings drawn from the variational autoencoder").show()

These are not sharper than the autoencoder's. They are softer, and with this model and this loss
that is no accident: the clouds of neighbouring buildings overlap, so nearly the same code has
to stand for buildings whose edges lie in different cells. Scored cell by cell, the safest answer
there is a probability in between, and a hedged black-and-white picture is grey at the edges. What it does have is
a place to draw from, and almost everything it produces is a plausible outline rather than an
hourglass.

That trade is the honest summary of the variational autoencoder. It buys a usable space at the
cost of sharpness. Session 7 shows the family of models that gets the sharpness back, and pays
for it in computing time.

## Temperature

The spread of the draw is a dial. Pull it down and you get the middle of the distribution: sober
rectangles, all alike. Pull it up and you draw from regions where few training codes lie. With a
decoder this small the result tends to get more extreme rather than broken, and whether it ever
turns into nonsense is something you check, not something you assume. It is the same dial that
appears next session on letters and in every image tool you have used.

In [ ]:
temperaturen = [0.2, 0.5, 0.8, 1.0, 1.4, 2.0]
gezogene = {t: ziehe(vae, 8, temperatur=t, wuerfel=torch.Generator().manual_seed(4))
            for t in temperaturen}

fig = make_subplots(rows=1, cols=8, horizontal_spacing=0.01)
for n in range(8):
    fig.add_trace(go.Heatmap(zmin=0, zmax=1, z=gezogene[1.0][n][::-1], colorscale="Greys", reversescale=True,
                             showscale=False), row=1, col=n + 1)
fig.frames = [go.Frame(name=str(t),
                       data=[go.Heatmap(zmin=0, zmax=1, z=gezogene[t][n][::-1], colorscale="Greys",
                                        reversescale=True, showscale=False)
                             for n in range(8)],
                       traces=list(range(8)),
                       layout=go.Layout(title=f"temperature {t}"))
              for t in temperaturen]
fig.update_layout(height=220, title="temperature 1.0",
                  sliders=[dict(active=temperaturen.index(1.0),
                                currentvalue=dict(prefix="temperature "),
                                steps=[dict(method="animate", label=str(t),
                                            args=[[str(t)],
                                                  dict(mode="immediate",
                                                       frame=dict(duration=0, redraw=True))])
                                       for t in temperaturen])])
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

## Walking, again

Session 4 walked between two buildings through the autoencoder and warned that some pairs
produce mush halfway across. Here is the same pair through both models.

In [ ]:
von, nach = 12, 260
schritte = np.linspace(0, 1, 9)
X = torch.tensor(bilder.reshape(len(bilder), -1), dtype=torch.float32)

reihen = []
with torch.no_grad():
    for name, encode, decode, quelle in (
            ("autoencoder", None, ae.decoder, codes_ae),
            ("vae", None, vae.decodiere, codes_vae)):
        a, b = quelle[von], quelle[nach]
        strecke = np.stack([(1 - t) * a + t * b for t in schritte]).astype("float32")
        reihen.append(torch.sigmoid(decode(torch.tensor(strecke))).numpy().reshape(-1, 32, 32))

fig = make_subplots(rows=2, cols=9, horizontal_spacing=0.006, vertical_spacing=0.08,
                    row_titles=["autoencoder", "vae"],
                    column_titles=[f"{t:.2f}" for t in schritte])
for zeile, reihe in enumerate(reihen, start=1):
    for spalte, bild in enumerate(reihe, start=1):
        fig.add_trace(go.Heatmap(zmin=0, zmax=1, z=bild[::-1], colorscale="Greys", reversescale=True,
                                 showscale=False), row=zeile, col=spalte)
fig.update_layout(height=320, title="the same two buildings, two ways across")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

## Asking for a period

The last step is to stop taking what comes and start asking. Feed the construction period into
the encoder and the decoder as an extra input, train the same way, and afterwards you can say
which kind of building you want.

Be careful with what follows. The dataset is badly unbalanced: 806 buildings from before 1918,
82 between the wars, 183 after 1946, of which only 19 are later than 1976. A model trained on that has
seen the first group ten times as often as the second, and it would be no surprise at all if the
three groups came out looking the same.

So we check instead of claiming. The next cell draws eight buildings per period. Look at them
before you read on, and decide for yourself whether you could sort them.

In [ ]:
fig = make_subplots(rows=len(GRUPPEN), cols=8, horizontal_spacing=0.01, vertical_spacing=0.06,
                    row_titles=GRUPPEN)
for zeile, name in enumerate(GRUPPEN):
    c = torch.zeros(8, len(GRUPPEN))
    c[:, zeile] = 1.0
    stapel = ziehe(bedingt, 8, c=c, wuerfel=torch.Generator().manual_seed(7))
    for spalte, bild in enumerate(stapel, start=1):
        fig.add_trace(go.Heatmap(zmin=0, zmax=1, z=bild[::-1], colorscale="Greys", reversescale=True,
                                 showscale=False), row=zeile + 1, col=spalte)
fig.update_layout(height=150 * len(GRUPPEN) + 60,
                  title="eight buildings asked for per period, same eight random draws each time")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

def seitenverhaeltnis(bild):
    # width over height of the box around the filled cells
    zeilen, spalten = np.nonzero((bild > 0.5).any(axis=1))[0], np.nonzero((bild > 0.5).any(axis=0))[0]
    return (spalten[-1] - spalten[0] + 1) / (zeilen[-1] - zeilen[0] + 1)


for n, name in enumerate(GRUPPEN):
    echte = bilder[gruppen == n]
    print(f"{name:10s} {len(echte):4d} buildings, average fill {echte.mean():.3f}, "
          f"average width/height {np.mean([seitenverhaeltnis(b) for b in echte]):.2f}")

Whatever you decided, hold on to the reasoning rather than the answer. If the three rows look
alike, possible explanations include that the periods do not differ much in outline, and that
the model did not learn to use the condition, for instance because 82 examples are few. Those
are different statements, and these rows alone do not separate them.

> A conditional model that ignores its condition looks exactly like one whose condition does not
> matter. These rows alone cannot tell you which you have.

## What this is not

Everything on this page is a black-and-white raster of an outline at 32 by 32. There are no
walls, no rooms, no doors, no structure, no orientation to the sun, no site. An outline that
comes out of this model is a suggestion about massing, and the distance from here to a building
is the part of the work you are studying for.

Say that out loud in front of a client before somebody else says it for you.

## Tests

In [ ]:
w1 = torch.Generator().manual_seed(3)
w2 = torch.Generator().manual_seed(3)
a = ziehe(vae, 6, wuerfel=w1)
b = ziehe(vae, 6, wuerfel=w2)
assert a.shape == (6, 32, 32), f"wrong shape: {a.shape}"
assert np.allclose(a, b), "same seed has to give the same buildings"
assert a.min() >= 0.0 and a.max() <= 1.0, "the output has to be squashed to 0..1"

kalt = ziehe(vae, 200, temperatur=0.1, wuerfel=torch.Generator().manual_seed(8))
heiss = ziehe(vae, 200, temperatur=2.0, wuerfel=torch.Generator().manual_seed(8))
assert kalt.std(axis=0).mean() < heiss.std(axis=0).mean(), (
    "a higher temperature has to produce more varied buildings")

c = torch.zeros(4, len(GRUPPEN))
c[:, 1] = 1.0
mit = ziehe(bedingt, 4, c=c, wuerfel=torch.Generator().manual_seed(2))
assert mit.shape == (4, 32, 32), "the conditional model wants a one-hot row per building"

fuell = ziehe(vae, 200, wuerfel=torch.Generator().manual_seed(5)).mean()
assert 0.1 < fuell < 0.6, f"drawn buildings are empty or solid: fill {fuell:.3f}"
print(f"all green, average fill of 200 drawn outlines {fuell:.3f}, "
      f"real buildings {bilder.mean():.3f}")

## Where you stand now

You can say why random codes drawn for a plain autoencoder need not decode into plausible
buildings, and show it rather than assert it.
You know what the variational version changes, in one sentence each for the two terms, and what
it costs. You can turn a temperature dial and predict which way the results will move. You have
asked a model for a category and checked, honestly, whether it listened.

Next session the outputs stop being pictures and become letters, the loss function changes for
the first time since session 1, and you will be able to say what happens inside the tool you use
every day.

---

### Terms to learn

| term | in one sentence |
|---|---|
| generative model | one that produces new examples instead of predicting a number |
| sampling | drawing a code at random and decoding it |
| prior | the distribution you are allowed to draw from; here a standard normal |
| variational autoencoder | encoder outputs a mean and a spread; a second loss term keeps the codes together |
| KL term | the part of the loss that pulls the codes towards the prior |
| temperature | how far from the middle of the distribution you draw |
| mode averaging | why VAE outputs are soft: covering several possibilities at once |
| conditional generation | asking for a category, by feeding it to encoder and decoder |
| class imbalance | one category dominating the training data, and quietly dominating the output |
| massing | an outline is a statement about volume, not a floor plan |

## Homework

Ten points, due before the next session: four for running it, six for your thoughts on it.

**How to do it**

1. Fill in the `YOUR TURN` gap above, yourself or from the solution notebook. If you took it from
   the solution, say so in one line at the top of your answers; that costs nothing.
2. Below, change only the lines under `# YOUR CHOICE`.
3. Write your answers into the last cell.
4. Choose Kernel, Restart Kernel and Run All Cells, check that everything runs, and upload the
   notebook in TUWEL.

**How it is graded**

- Four points for the run: it runs without an error (one), it shows what the task asks for (one),
  and every number you give about your run appears in your output or is worked out from it in a
  step you write down (two). Numbers the questions give you, examples from outside the notebook
  and forecasts you mark as your own expectation are exempt.
- Six points for your answers, one per question, no halves. Say what you saw and what it means,
  and you get the point, also with borrowed code. A sentence that would fit any model gets none.

### The task

1. Choose a seed of your own and draw ten outlines. For each one, one sentence: would it do as
   a starting point for a design, and why. "None of the ten would, and this is why" is a
   complete answer, as long as the reason is something you can see in all ten.
2. Walk the temperature up with the same seed and check whether, within the temperatures you
   try, a point appears where variety turns into nonsense. Finding none is a result too.
3. Say what an outline from this model lacks on the way to a floor plan.

In [ ]:
# YOUR CHOICE: a whole number as your seed, and a temperature above 0. Same seed, same ten
# outlines, so your text keeps matching the pictures.
mein_samen = None
meine_temperatur = 1.0

assert isinstance(mein_samen, int), "enter a whole number as your seed"
assert 0 < meine_temperatur <= 10, "a temperature above 0 and at most 10"

echt = (bilder > 0.5).reshape(len(bilder), -1)


def zum_naechsten(stapel):
    # for each outline: in how many cells it differs from the most similar real Vienna outline
    s = (stapel > 0.5).reshape(len(stapel), -1)
    return (s[:, None, :] != echt[None]).sum(axis=2).min(axis=1)


def quadratisch(fig):
    for name in fig.layout:
        if name.startswith("yaxis"):
            fig.layout[name].scaleanchor = "x" + name[5:]
    return fig


zehn = ziehe(vae, 10, temperatur=meine_temperatur,
             wuerfel=torch.Generator().manual_seed(mein_samen))
fig = make_subplots(rows=2, cols=5, horizontal_spacing=0.02, vertical_spacing=0.1,
                    subplot_titles=[f"outline {n}" for n in range(10)])
for n, bild in enumerate(zehn):
    r, c = divmod(n, 5)
    fig.add_trace(go.Heatmap(z=bild[::-1], zmin=0, zmax=1, colorscale="Greys", reversescale=True,
                             showscale=False), row=r + 1, col=c + 1)
fig.update_layout(height=430, title=f"ten outlines, temperature {meine_temperatur}, "
                                    f"seed {mein_samen}")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
quadratisch(fig).show()

for n, (bild, weg) in enumerate(zip(zehn, zum_naechsten(zehn))):
    print(f"outline {n}: filled {(bild > 0.5).mean():.3f}, "
          f"differs from the most similar real outline in {weg:3d} cells")
print(f"real buildings: filled {(bilder > 0.5).mean():.3f}")
print("filled: the share of the 1024 cells that come out darker than half grey")

In [ ]:
# YOUR CHOICE: three to ten temperatures, rising, each above 0 and at most 10
leiter = [0.3, 0.6, 1.0, 1.5, 2.5, 4.0]

assert 3 <= len(leiter) <= 10 and all(0 < a < b for a, b in zip(leiter, leiter[1:])), (
    "three to ten temperatures, rising, all above 0")
assert leiter[-1] <= 10, "at most 10"
reihen = [ziehe(vae, 6, temperatur=t, wuerfel=torch.Generator().manual_seed(mein_samen))
          for t in leiter]
fig = make_subplots(rows=len(leiter), cols=6, horizontal_spacing=0.01, vertical_spacing=0.02,
                    row_titles=[f"T {t}" for t in leiter])
for zeile, stapel in enumerate(reihen, start=1):
    for spalte, bild in enumerate(stapel, start=1):
        fig.add_trace(go.Heatmap(z=bild[::-1], zmin=0, zmax=1, colorscale="Greys", reversescale=True,
                                 showscale=False), row=zeile, col=spalte)
fig.update_layout(height=130 * len(leiter) + 80, title="the same six draws, hotter and hotter")
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
quadratisch(fig).show()

for t in leiter:
    viele = ziehe(vae, 100, temperatur=t, wuerfel=torch.Generator().manual_seed(mein_samen))
    print(f"T {t}: 100 draws, filled {(viele > 0.5).mean():.3f}, differ from the most similar "
          f"real outline in {np.median(zum_naechsten(viele)):.0f} cells (median)")
print("median: half of the 100 draws are closer to a real outline than this, half further")

### Your interpretation

1. **Size.** How many of your ten would you keep? Compare their fill with the real buildings. Then say what the cell count to the most similar real outline does tell you, and what it cannot tell you about whether the model invented or copied.
2. **Plausibility.** Your ten one-sentence verdicts, numbered as in the figure, or one verdict for all ten. Each must name something you can see.
3. **Units.** What does a temperature of 1.0 mean for this model, and what does 2.0 mean? Answer with the distribution the codes are drawn from, and what the second term in the loss did during training.
4. **Range.** Within the temperatures you tried, does variety turn into nonsense somewhere? Back your answer with the figure and the printed numbers. If you find no such point, say so, and say what that tells you about this decoder.
5. **One source of error.** The outlines are soft at the edges. Say why, in terms of what the decoder was trained to do.
6. **Design consequence.** What does an outline from this model lack on the way to a floor plan? Name at least four things you would have to add, in the order you would add them, and say why that order: what does each step need from the one before?

*Your text here.*